In [ ]:
# import necessary packages
# import necessary packages
# %pip install folium
import folium
import platform
from pathlib import Path
import pandas as pd
import geopandas as gpd
import seaborn as sns
import pygris
import calendar
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
from matplotlib.patches import Rectangle
import pyarrow



FIRE

In [ ]:
# establish paths to datasets - fire
project_fldr = Path.cwd().parents[1]
raw_fldr = project_fldr / 'data' / 'raw'
processed_fldr = project_fldr / 'data' / 'processed'
figure_fldr = project_fldr/"figures"

# read in the processed fire dataset
fire_path = processed_fldr / "fire_shp_25" / "fire_shp_25.shp"
fire = gpd.read_file(fire_path)


# crs for folium
fire = fire.to_crs("EPSG:4326")

WETLANDS

In [ ]:
# connect to box
# establish paths to datasets - wetlands

# set a path to the main box folder based on machine's os
# if windows: 
if platform.system() == 'Windows':
    box_path = Path.home() / 'Box'
# if mac:
else:
    box_path = Path.home() / "Library" / "CloudStorage" / "Box-Box"

#Confirm that the box path is valid
if box_path.exists():
    print(f'Box folder found at {box_path}')
else:
    print(f'Failed to locate the Box folder.\nIs the BoxDrive app running?\nAre you logged in?')

# create path object that goes to the specific folder on Box drive
wetlands_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' / 'wetlands_subset' / 'wetlands_subset.geojson'
print(wetlands_path.exists())

In [ ]:
# read in the wetlands geojson file, this will take a WHILE - 10 mins 1st try 
wetlands = gpd.read_file(wetlands_path, engine="pyogrio")


In [ ]:
# make a separate copy for mapping to preserve initial df

wetlands_map = wetlands.copy()

wetlands_map.shape

In [ ]:
# ensure valid geometries to then simplify the data for display
print(wetlands_map.geometry.is_valid.value_counts())

# 227 are false
wetlands_map["geometry"] = wetlands_map.geometry.make_valid()

# check for validity again
# print(wetlands.geometry.is_valid.value_counts()) 
# this worked, 


In [ ]:
# create subsets for each of the 5 types of systems 
# first preserve only some columns

cols = ['C System','Hectares', 'geometry']

wetlands_map = wetlands_map[cols]
wetlands_map.columns

In [ ]:
# subset with loc

marine = wetlands_map.loc[ lambda df: df["C System"] == "Marine"]
estuarine = wetlands_map.loc[ lambda df: df["C System"] == "Estuarine"]
lacustrine = wetlands_map.loc[ lambda df: df["C System"] == "Lacustrine"]
riverine = wetlands_map.loc[ lambda df: df["C System"] == "Riverine"]
palustrine = wetlands_map.loc[ lambda df: df["C System"] == "Palustrine"]




In [ ]:
## skipping riverine and palustrine because they are too large

In [ ]:
# define functions to perform actions
def simpler(df, name):
    df_simpler = df.copy()
    df_simpler["geometry"] = df.geometry.simplify(
        tolerance=100,
        preserve_topology=True
        )
    return df_simpler


#chatgpt helped to finetune this code
def vertices_before(df):
    # count vertices 
    def count_vertices(df):
        if df.geom_type == "Polygon":
            return len(df.exterior.coords) + sum(
                len(ring.coords) for ring in df.interiors
            )

        elif df.geom_type == "MultiPolygon":
            return sum(count_vertices(poly) for poly in df.geoms)

        return 0
    return print("vertices before:", df.geometry.apply(count_vertices).sum())

def vertices_after(df):
        # count vertices after
    def count_vertices(df):
        if df.geom_type == "Polygon":
            return len(df.exterior.coords) + sum(
                len(ring.coords) for ring in df.interiors
            )

        elif df.geom_type == "MultiPolygon":
            return sum(count_vertices(poly) for poly in df.geoms)

        return 0
    return print("vertices after:", df.geometry.apply(count_vertices).sum())


In [ ]:
# marine simplification
print("MARINE")
vertices_before(marine)
marine_simpler = simpler(marine, "marine")
vertices_after(marine_simpler)



In [ ]:
# estuarine simplification
print("ESTUARINE")
vertices_before(estuarine)
estuarine_simpler = simpler(estuarine, "estuarine")
vertices_after(estuarine_simpler)



In [ ]:
# lacustrine simplification
print("LACUSTRINE")
vertices_before(lacustrine)
lacustrine_simpler = simpler(lacustrine, "lacustrine")
vertices_after(lacustrine_simpler)



In [ ]:
SKIP
# riverine simplification
print("RIVERINE")
vertices_before(riverine)
riverine_simpler = simpler(riverine, "riverine")
vertices_after(riverine_simpler)



In [ ]:
SKIP
# palustrine simplification
print("PALUSTRINE")
vertices_before(palustrine)
palustrine_simpler = simpler(palustrine, "palustrine")
vertices_after(palustrine_simpler)

If DISSOLVING (don't do this)


In [ ]:
# dissolve by wetland system to get 5 polygons total
#wetlands_dissolved = wetlands_map.dissolve(
    by="C System"
).reset_index()

In [ ]:
#wetlands_dissolved.head()
wetlands_dissolved.info()

In [ ]:
#simplify the polygons to make easier to save
#wetlands_dissolved_simple = wetlands_dissolved.to_crs(epsg=26910).copy()

#wetlands_dissolved_simple["geometry"] = (
    wetlands_dissolved_simple.geometry.simplify(
        tolerance=100,
        preserve_topology=True
    )
)

In [ ]:
# do a local save as a first try because other versions dont work
#wetlands_dissolved.to_parquet("wetlands_dissolved.parquet")
# FAILED METHODS
#wetlands_dissolved.to_file("wetlands_dissolved.geojson", driver= "GeoJSON")

#wetlands_dissolved_path = box_path / 'ENVIRON 712L-Students' / 'Data' / 'BunkerAndersonTseng' 

#wetlands_dissolved.to_file(wetlands_dissolved_path/"wetlands_dissolved.geojson", driver = "GeoJSON")

## SAVE WETLANDS DISSOLVED this did not work and crashed the kernel
## wetlands_dissolved.to_file(processed_fldr/"wetlands_dissolved.geojson", driver = "GeoJSON")

If simplifying (process still takes a WHILE)

In [ ]:
# count the total vertices before simplification
def count_vertices(geom):
    if geom.geom_type == "Polygon":
        return len(geom.exterior.coords) + sum(
            len(ring.coords) for ring in geom.interiors
        )

    elif geom.geom_type == "MultiPolygon":
        return sum(count_vertices(poly) for poly in geom.geoms)

    return 0

vertices_before = wetlands.geometry.apply(count_vertices).sum()

print("Vertices before:", vertices_before)

In [ ]:
# before converting to the necessary folium one, leave it in projected meters crs for simplification


# simplify with a tolerance of 10
wetlands_map["geometry"] = wetlands_map.geometry.simplify(
    tolerance=100,
    preserve_topology=True
)

# count vertices after
vertices_after = wetlands_map.geometry.apply(count_vertices).sum()

print("Vertices after:", vertices_after)

In [ ]:
# calculate percent reducation
reduction = (1 - vertices_after / vertices_before) * 100

print(f"Vertex reduction: {reduction:.2f}%")

In [ ]:
# convert to necessary folium CRS
wetlands_map = wetlands_map.to_crs("EPSG:4326")

#double check validity
print(wetlands_map.geometry.is_valid.value_counts())

In [ ]:
# repair geometries again
wetlands_map["geometry"] = wetlands_map.geometry.make_valid()


# BACK TO MAPPING

CALIFORNIA

In [ ]:
# make map just for CA
state_outlines = pygris.states()
ca_outline = state_outlines[state_outlines['NAME'] == 'California']
#folium crs
ca_outline = ca_outline.to_crs("EPSG:4326")



In [ ]:
# DEFINE ASPECTS OF THE MAP

# bounding coordinates for California and rest of map
minx, miny, maxx, maxy = ca_outline.total_bounds


#plot wetlands, by system color
system_colors = {
    "Marine": "#02518a",
    "Estuarine": "#0b2ce9",
    "Riverine": "#17becf",
    "Lacustrine": "#01f694",
    "Palustrine": "#4bcb00"
}

In [ ]:
# add in wetlands by first subsetting them

# NEED TO USE SIMPLER BECAUSE OF VERTICES SIMPLIFICATION AND NEW DF

## put in only Marine Estuarine and Lacustrine to 
# start because smaller datasets will be easier to develop, can add in the other two at a later date

marine_simpler = marine_simpler.to_crs("EPSG:4326")
estuarine_simpler = estuarine_simpler.to_crs("EPSG:4326")
lacustrine_simpler = lacustrine_simpler.to_crs("EPSG:4326")

In [ ]:
# MAKE THE EXPLORE

# create the actual map as a base now
m = folium.Map(
    zoom_start=6,
    tiles=None
)

# ESRI satellite basemap
folium.TileLayer(
    tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
    attr="Esri, Maxar, Earthstar Geographics, and the GIS User Community",
    name="Satellite Imagery",
    overlay=False,
    control=False
).add_to(m)

# this should leave us so that the initial zoom is the outline of california
m.fit_bounds([
    [miny, minx],  # Southwest corner
    [maxy, maxx]   # Northeast corner
])

# add CA outline
folium.GeoJson(
    ca_outline.to_json(),
    style_function=lambda x: {
        "color": "black",
        "fillOpacity": 0,
        "weight": 2
    },
    control= False # always on
).add_to(m)

# Marine layer
folium.GeoJson(
    marine_simpler.to_json(),
    name="Marine System Wetlands",
    style_function=lambda x: {
        "color": system_colors["Marine"],
        "fillColor": system_colors["Marine"],
        "fillOpacity": 0.9,
        "weight": 0.5
    },
    control=True #can be toggled
).add_to(m)

# Estuarine layer
folium.GeoJson(
    estuarine_simpler.to_json(),
    name="Estuarine System Wetlands",
    style_function=lambda x: {
        "color": system_colors["Estuarine"],
        "fillColor": system_colors["Estuarine"],
        "fillOpacity": 0.9,
        "weight": 0.5
    },
    control=True #can be toggled
).add_to(m)

# Lacustrine layer
folium.GeoJson(
    lacustrine_simpler.to_json(),
    name="Lacustrine System Wetlands",
    style_function=lambda x: {
        "color": system_colors["Lacustrine"],
        "fillColor": system_colors["Lacustrine"],
        "fillOpacity": 0.9,
        "weight": 0.5
    },
    control=True #can be toggled
).add_to(m)

# Add layer control
folium.LayerControl(collapsed=False).add_to(m)

m


In [ ]:
# save the simplified files to call more easily
marine_simpler.to_file(
    processed_fldr/"marine_simpler.geojson", driver= "GeoJSON")

estuarine_simpler.to_file(
    processed_fldr/"estuarine_simpler.geojson", driver= "GeoJSON")

lacustrine_simpler.to_file(
    processed_fldr/"lacustrine_simpler.geojson", driver= "GeoJSON")

ca_outline.to_file(
    processed_fldr/"ca_outline.geojson", driver= "GeoJSON")